In [29]:
from openai import OpenAI
import pandas as pd
import os
import numpy as np 
import pandas as pd
import minsearch
from dotenv import load_dotenv
from tqdm.auto import tqdm
import json

In [30]:
load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [31]:
data = pd.read_csv('data/Mental_Health_FAQ.csv', index_col=False)

In [32]:
data.head()

,Question_ID,Questions,Answers
0,1590140,What does it mean to have a mental illness?,Mental illnesses are health conditions that di...
1,2110618,Who does mental illness affect?,It is estimated that mental illness affects 1 ...
2,6361820,What causes mental illness?,It is estimated that mental illness affects 1 ...
3,9434130,What are some of the warning signs of mental i...,Symptoms of mental health disorders vary depen...
4,7657263,Can people with mental illness recover?,"When healing from mental illness, early identi..."


In [33]:
data = data.rename(columns={'Question_ID': 'question_id', 'Questions': 'questions', 'Answers': 'answers'})
data.head()

,question_id,questions,answers
0,1590140,What does it mean to have a mental illness?,Mental illnesses are health conditions that di...
1,2110618,Who does mental illness affect?,It is estimated that mental illness affects 1 ...
2,6361820,What causes mental illness?,It is estimated that mental illness affects 1 ...
3,9434130,What are some of the warning signs of mental i...,Symptoms of mental health disorders vary depen...
4,7657263,Can people with mental illness recover?,"When healing from mental illness, early identi..."


In [34]:
data.duplicated(subset=['questions']).sum()

np.int64(0)

In [35]:
data.dtypes

question_id     int64
questions      object
answers        object
dtype: object

In [36]:
documents= data.to_dict(orient='records')
documents[0]

{'question_id': 1590140,
 'questions': 'What does it mean to have a mental illness?',
 'answers': 'Mental illnesses are health conditions that disrupt a personâ€™s thoughts, emotions, relationships, and daily functioning. They are associated with distress and diminished capacity to engage in the ordinary activities of daily life.\nMental illnesses fall along a continuum of severity: some are fairly mild and only interfere with some aspects of life, such as certain phobias. On the other end of the spectrum lie serious mental illnesses, which result in major functional impairment and interference with daily life. These include such disorders as major depression, schizophrenia, and bipolar disorder, and may require that the person receives care in a hospital.\nIt is important to know that mental illnesses are medical conditions that have nothing to do with a personâ€™s character, intelligence, or willpower. Just as diabetes is a disorder of the pancreas, mental illness is a medical condit

In [37]:
template="""You are emulating a patient who is concerned about mental health.
Based on the answer provided, formulate 5 questions that this patient might ask. 
The questions should be complete, not too short, and use as few words as possible from the original answer.

The record:
Question_ID:{question_id}
Questions: {questions}
Answer: {answers}

Provide the output in parsable JSON format without using code blocks:

{{["question1", "question2",..., "question5"]}}

""".strip()

In [38]:
prompt= template.format(**documents[0])

In [39]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [40]:
def generate_questions(documents):
    prompt = template.format(**documents)

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )

    json_response = response.choices[0].message.content
    return json_response

In [41]:
results={}
for doc in tqdm(documents): 
    doc_id = doc['question_id']
    if doc_id in results:
        continue

    questions_raw = generate_questions(doc)
    questions = json.loads(questions_raw)
    results[doc_id] = questions['questions']

  0%|          | 0/98 [00:00<?, ?it/s]

In [42]:
results_ = []

for doc_id, questions in results.items():
    for q in questions:
        results_.append((doc_id, q))

In [43]:
results_[12]

(6361820,
 'Why are young people particularly vulnerable to mental health conditions?')

In [44]:
df = pd.DataFrame(results_, columns=['id', 'question'])

In [45]:
df.head(10)

,id,question
0,1590140,How do mental conditions affect daily life and...
1,1590140,What is the range of severity for mental illne...
2,1590140,What serious disorders require hospital care?
3,1590140,How are mental illnesses similar to physical h...
4,1590140,What treatments are effective for managing men...
5,2110618,How many adults in America are affected by men...
6,2110618,Can you explain how mental illness impacts dif...
7,2110618,What age groups are particularly vulnerable to...
8,2110618,Why is it challenging to identify mental healt...
9,2110618,What signs should parents look for to detect m...


In [52]:
df.to_csv('../mentalhealthqa/data/ground_truth_data.csv', index=False)